In [ ]:
import pandas as pd
import geopandas as gpd
import osmnx as ox

In [ ]:
def load_flood_snapshot(path, label, timestamp):
    df = pd.read_csv(path)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.x, df.y),
        crs="EPSG:4326",
    )
    return {"label": label, "timestamp": pd.to_datetime(timestamp), "gdf": gdf}

In [ ]:
# Load flood grid data for multiple time snapshots (scalable t1, t2, t3, ...)
flood_snapshots = [
    {
        "label": "s0",
        "path": "../data/raw/flood/so.csv",
        "timestamp": "2025-11-21 6:00:00",
    },
    {
        "label": "s1",
        "path": "../data/raw/flood/bc5_20251124_2200.csv",
        "timestamp": "2025-11-24 22:00:00",
    },
]

snapshots = [load_flood_snapshot(**snap) for snap in flood_snapshots]

graph = ox.load_graphml("../data/processed/hatyai_graph_ready_for_flood.graphml")
nodes, edges = ox.graph_to_gdfs(graph)

edges = edges.reset_index()
nodes_proj = nodes.to_crs(epsg=32647)

nodes_with_flood = nodes.drop(columns="geometry").copy()

for snap in snapshots:
    label = snap["label"]
    distance_col = f"flood_distance_{label}_m"

    right_proj = snap["gdf"][['gridcode', 'geometry']].to_crs(epsg=32647)

    joined = gpd.sjoin_nearest(
        nodes_proj,
        right_proj,
        how="left",
        distance_col=distance_col,
    ).rename(columns={"gridcode": f"flood_level_{label}"})

    joined = (
        joined.sort_values(distance_col)
        .groupby(level=0)
        .first()
        .reindex(nodes_with_flood.index)
    )

    nodes_with_flood[f"flood_level_{label}"] = joined[f"flood_level_{label}"].to_numpy()
    nodes_with_flood[distance_col] = joined[distance_col].to_numpy()

edges_with_flood = edges.copy()

for snap in snapshots:
    label = snap["label"]

    node_levels = nodes_with_flood.groupby("osmid")[f"flood_level_{label}"].max()

    u_level = edges_with_flood["u"].map(node_levels)
    v_level = edges_with_flood["v"].map(node_levels)

    edges_with_flood[f"flood_level_{label}"] = pd.concat(
        [u_level, v_level], axis=1
    ).max(axis=1)

nodes_with_flood.to_csv("../data/processed/hatyai_nodes_flood_multi.csv", index=False)
edges_with_flood.to_csv("../data/processed/hatyai_edges_flood_multi.csv", index=False)

print("Flood snapshots attached (scalable):")
print("Snapshot labels:", [s["label"] for s in snapshots])
print(nodes_with_flood.filter(regex="flood_level_.*").head())
print(edges_with_flood.filter(regex="flood_level_.*").head())
